In [1]:
pip install langchain_huggingface

  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached huggingface_hub-1.21.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached tqdm-4.68.3-py3-none-any.whl.metadata (57 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)
Using cached huggingface_hub-1.21.0-py3-none-any.


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

C:\Users\maaz7\AppData\Local\Temp\ipykernel_41580\3849627479.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
load_dotenv()


True

In [4]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
loader = PyPDFLoader('')
docs = loader.load()



In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=10)
chunks = splitter.split_documents(docs)

In [ ]:
len(chunks)

In [ ]:
embeddings = HuggingFaceEmbeddings(model = "BAAI/bge-large-en-v1.5")
vector_store = FAISS(chunks, embeddings)


In [ ]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={'k':4})

In [ ]:
@tool
def rag_tool(query):
    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual / conceptual questiona
    that might be answeredd from the stored documents.
    """
    result = retriever.invoke(query)
    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]
    return {
        'query':query,
        'context':context,
        'metadata':metadata
    }

In [ ]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

NameError: name 'rag_tool' is not defined

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    messages = state['messages']
    res = llm_with_tools.invoke(messages)
    return {'messages':[res]}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(ChatState)

graph.add_node("chat_node",chat_node)
graph.add_node("tools",tool_node)
graph.add_edge(START,"chat_node")
graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge("tools","chat_node")
chatbot = graph.compile()

In [ ]:
res = chatbot.invoke(
    {
        'messages':[
            HumanMessage(content=(""))
        ]
        
    }
)

In [ ]:
print(res["messages"][-1].content)